In [ ]:


import pandas as pd
import numpy as np
from google.colab import files


uploaded = files.upload()
file_name = next(iter(uploaded))


df = pd.read_csv(file_name)



before_rows, before_columns = df.shape
before_duplicates = df.duplicated().sum()
before_missing = df.isnull().sum().sum()


missing_before = pd.DataFrame({
    "Missing Before": df.isnull().sum(),
    "Percentage Before": (df.isnull().sum() / len(df)) * 100
})

print("========== BEFORE CLEANING ==========")
print("Rows:", before_rows)
print("Columns:", before_columns)
print("Duplicate Rows:", before_duplicates)
print("Total Missing Values:", before_missing)

print("\n========== MISSING VALUES BEFORE CLEANING ==========")
display(
    missing_before[missing_before["Missing Before"] > 0].round(2)
)




df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(r"[^a-zA-Z0-9_]", "", regex=True)
)


original_columns = df.columns.tolist()


df = df.drop_duplicates()


text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})


df["patient_admission_date"] = pd.to_datetime(
    df["patient_admission_date"],
    dayfirst=True,
    errors="coerce"
)


df["patient_admission_time"] = pd.to_datetime(
    df["patient_admission_time"],
    format="%I:%M:%S %p",
    errors="coerce"
)


df["patient_admission_date"] = df["patient_admission_date"].fillna(
    df["patient_admission_date"].mode()[0]
)

df["patient_admission_time"] = df["patient_admission_time"].fillna(
    df["patient_admission_time"].mode()[0]
)


df["patient_age"] = pd.to_numeric(df["patient_age"], errors="coerce")
df["patient_waittime"] = pd.to_numeric(df["patient_waittime"], errors="coerce")
df["patient_satisfaction_score"] = pd.to_numeric(
    df["patient_satisfaction_score"],
    errors="coerce"
)


df.loc[df["patient_age"] < 0, "patient_age"] = np.nan
df.loc[df["patient_waittime"] < 0, "patient_waittime"] = np.nan


df.loc[
    ~df["patient_satisfaction_score"].between(1, 5),
    "patient_satisfaction_score"
] = np.nan


numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())


text_columns = df.select_dtypes(include=["object", "string"]).columns

for col in text_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])


df["patient_gender"] = df["patient_gender"].str.lower()
df["patient_race"] = df["patient_race"].str.title()
df["department_referral"] = df["department_referral"].str.title()
df["patient_admission_flag"] = df["patient_admission_flag"].str.lower()

df["admission_year"] = df["patient_admission_date"].dt.year
df["admission_month"] = df["patient_admission_date"].dt.month
df["admission_day"] = df["patient_admission_date"].dt.day
df["admission_day_name"] = df["patient_admission_date"].dt.day_name()
df["admission_hour"] = df["patient_admission_time"].dt.hour


df["age_group"] = pd.cut(
    df["patient_age"],
    bins=[0, 18, 35, 50, 65, 120],
    labels=["Child", "Young Adult", "Adult", "Senior", "Elderly"],
    include_lowest=True
)

df["waittime_category"] = pd.cut(
    df["patient_waittime"],
    bins=[-1, 15, 30, 60, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)


df["satisfaction_category"] = pd.cut(
    df["patient_satisfaction_score"],
    bins=[0, 2, 3, 4, 5],
    labels=["Poor", "Average", "Good", "Excellent"],
    include_lowest=True
)


df["is_admitted"] = (
    df["patient_admission_flag"] == "admission"
).astype(int)



after_rows, after_columns = df.shape
after_duplicates = df.duplicated().sum()
after_missing = df.isnull().sum().sum()


missing_after = pd.DataFrame({
    "Missing After": df.isnull().sum(),
    "Percentage After": (df.isnull().sum() / len(df)) * 100
})


missing_comparison = pd.concat(
    [missing_before, missing_after],
    axis=1
).fillna(0)

missing_comparison["Missing Values Reduced"] = (
    missing_comparison["Missing Before"] -
    missing_comparison["Missing After"]
)

missing_comparison["Percentage Reduced"] = (
    missing_comparison["Percentage Before"] -
    missing_comparison["Percentage After"]
)


new_columns = [col for col in df.columns if col not in original_columns]


comparison_summary = pd.DataFrame({
    "Metric": [
        "Number of Rows",
        "Number of Columns",
        "Duplicate Rows",
        "Total Missing Values",
        "New Columns Added"
    ],
    "Before Cleaning": [
        before_rows,
        before_columns,
        before_duplicates,
        before_missing,
        0
    ],
    "After Cleaning": [
        after_rows,
        after_columns,
        after_duplicates,
        after_missing,
        len(new_columns)
    ],
    "Difference": [
        after_rows - before_rows,
        after_columns - before_columns,
        after_duplicates - before_duplicates,
        after_missing - before_missing,
        len(new_columns)
    ]
})



print("\n========== BEFORE VS AFTER FULL COMPARISON ==========")
display(comparison_summary)

print("\n========== MISSING VALUES: BEFORE VS AFTER ==========")
display(
    missing_comparison[
        (missing_comparison["Missing Before"] > 0) |
        (missing_comparison["Missing After"] > 0)
    ].round(2)
)

print("\n========== COLUMNS WITH MISSING PERCENTAGE >= 2% ==========")
display(
    missing_after[
        missing_after["Percentage After"] >= 2
    ].round(2)
)

print("\n========== NEW COLUMNS ADDED ==========")
print(new_columns)

print("\n========== CLEANED DATA PREVIEW ==========")
display(df.head())


df.to_csv("cleaned_healthcare_patient_flow_data.csv", index=False)
files.download("cleaned_healthcare_patient_flow_data.csv")

Saving healthcare_analytics_patient_flow_data.csv to healthcare_analytics_patient_flow_data.csv
========== BEFORE CLEANING ==========
Rows: 9216
Columns: 11
Duplicate Rows: 0
Total Missing Values: 12099

========== MISSING VALUES BEFORE CLEANING ==========


,Missing Before,Percentage Before
Department Referral,5400,58.59
Patient Satisfaction Score,6699,72.69



========== BEFORE VS AFTER FULL COMPARISON ==========


,Metric,Before Cleaning,After Cleaning,Difference
0,Number of Rows,9216,9216,0
1,Number of Columns,11,20,9
2,Duplicate Rows,0,0,0
3,Total Missing Values,12099,0,-12099
4,New Columns Added,0,9,9



========== MISSING VALUES: BEFORE VS AFTER ==========


...

,Missing After,Percentage After



========== NEW COLUMNS ADDED ==========
['admission_year', 'admission_month', 'admission_day', 'admission_day_name', 'admission_hour', 'age_group', 'waittime_category', 'satisfaction_category', 'is_admitted']

========== CLEANED DATA PREVIEW ==========


,patient_id,patient_admission_date,patient_admission_time,merged,patient_gender,patient_age,patient_race,department_referral,patient_admission_flag,patient_satisfaction_score,patient_waittime,admission_year,admission_month,admission_day,admission_day_name,admission_hour,age_group,waittime_category,satisfaction_category,is_admitted
0,780-96-6113,2024-09-09,1900-01-01 09:25:00,W. Breede,female,63.0,African American,General Practice,not admission,5.0,32.0,2024,9,9,Monday,9,Senior,High,Excellent,0
1,714-35-6722,2024-09-09,1900-01-01 16:42:00,Y. Baldetti,male,31.0,Asian,Orthopedics,not admission,3.0,22.0,2024,9,9,Monday,16,Young Adult,Medium,Average,0
2,571-85-3714,2024-09-09,1900-01-01 00:14:00,M. Semerad,male,75.0,White,General Practice,not admission,3.0,16.0,2024,9,9,Monday,0,Elderly,Medium,Average,0
3,404-43-9499,2024-09-09,1900-01-01 20:33:00,K. Blaydes,male,79.0,African American,General Practice,admission,3.0,38.0,2024,9,9,Monday,20,Elderly,High,Average,1
4,552-51-5855,2024-09-09,1900-01-01 19:25:00,F. Dickerson,female,24.0,African American,General Practice,admission,3.0,36.0,2024,9,9,Monday,19,Young Adult,High,Average,1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>